# RGB Vegetation Mapping — Clean Demonstration

ExG and VARI from RGB imagery. Low-index regions are **heuristics**, not verified disease/stress labels. Video maps use **synthetic GPS coordinates**.

## 1. Setup
Place a shareable RGB image at `data/sample.jpg` or change `IMAGE_PATH` below. Run from the repository root.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "scripts"))
from vegetation_mapping import indices, clipped, low_index_mask, save_preview, synthetic_coordinates
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 2. Load an RGB image

In [ ]:
IMAGE_PATH = Path("data/sample.jpg")
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Add your own RGB image at {IMAGE_PATH}")
rgb = np.asarray(Image.open(IMAGE_PATH).convert("RGB"))
print("Image shape:", rgb.shape)

## 3. ExG, VARI and low-index overlay

ExG = 2G − R − B; VARI = (G − R)/(G + R − B). The 20th-percentile threshold is exploratory and image-relative.

In [ ]:
exg, vari = indices(rgb)
mask = low_index_mask(exg, vari)
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(rgb); ax[0].set_title("RGB")
ax[1].imshow(clipped(exg), cmap="RdYlGn"); ax[1].set_title("ExG")
ax[2].imshow(clipped(vari), cmap="RdYlGn"); ax[2].set_title("VARI")
ax[3].imshow(rgb)
ax[3].imshow(np.ma.masked_where(~mask, mask), cmap="Reds", alpha=0.45, vmin=0, vmax=1)
ax[3].set_title("Low-index overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Save results

In [ ]:
out = Path("outputs"); out.mkdir(exist_ok=True)
stats = save_preview(rgb, out / "vegetation_preview.png")
print(stats)

## 5. Optional: video processing

To extract frames every 0.2 seconds, compute per-frame metrics, and plot an illustrative simulated-coordinate map, run from the repository root:

`python scripts/vegetation_mapping.py --video data/drone_video.mp4 --interval 0.2`

**The coordinates are synthetic, not measurements.**